# MF-HGNN ABIDE_SiteLeaveGroupOut Reproduction Guide

This notebook includes:

1. Data loading and a brief overview
2. Dataset splitting (site-wise leave-group-out cross-validation) and summary
3. Full training procedure (calling `main.py`)
4. Evaluation with the best saved models and viewing logs

All core training and evaluation logic is identical to the original experimental code used in the paper, implemented via `main.py`, `dataload.py`, `model.py`, etc.


In [1]:
# Environment and dependencies

import os
import numpy as np
import torch

from opt import OptInit
from dataload import dataloader

# Initialize hyperparameters (same as running main.py from the command line)
opt = OptInit().initialize()
print('Device:', opt.device)

# Print several important paths for this notebook
print('subject_IDs_path:', opt.subject_IDs_path)
print('phenotype_path:', opt.phenotype_path)
print('data_path:', opt.data_path)
print('log_path:', opt.log_path)
print('ckpt_path:', opt.ckpt_path)


 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is train.
 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is

## 1. Data loading and overview

In this section, we use the `dataloader` class in `dataload.py` to reproduce exactly the same data-loading pipeline as in the official experiments, and briefly inspect key data structures:

- Raw node features `raw_features`
- Label vector `y`
- Phenotypic data `phonetic_data` (site, sex, age, etc.)
- Dynamic functional connectivity `dynamic_fc`


In [2]:
# 1. Data loading (detailed prints)

dl = dataloader()
raw_features, y, nonimg, phonetic_score, time_series, dynamic_fc = dl.load_data()

# Avoid flooding the notebook output; only show parts of large arrays
np.set_printoptions(threshold=20)

# Basic information
print('Number of raw_features samples:', len(raw_features))
print('Shape of labels y:', np.array(y).shape)
print('Shape of non-image phenotypes nonimg:', np.array(nonimg).shape)
print('Number of dynamic_fc samples:', len(dynamic_fc))

# Inspect the first sample
first_feat = raw_features[0]
print('\nType of the first sample:', type(first_feat))
if hasattr(first_feat, 'shape'):
    print('Shape of the first sample features:', first_feat.shape)
elif hasattr(first_feat, 'x'):
    # For torch_geometric.data.Data
    print('Shape of node features x of the first sample:', first_feat.x.shape)
    print('Shape of edge_index of the first sample:', first_feat.edge_index.shape)

# Label distribution
unique, counts = np.unique(y, return_counts=True)
print('\nLabel distribution (label: count):')
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

# Phenotypic information (SITE_ID, SEX, AGE_AT_SCAN)
print('\nSITE_ID for all samples:')
print(phonetic_score['SITE_ID'])

print('\nSEX for all samples:')
print(phonetic_score['SEX'])

print('\nAGE_AT_SCAN for all samples:')
print(phonetic_score['AGE_AT_SCAN'])


Number of raw_features samples: 871
Shape of labels y: (871,)
Shape of non-image phenotypes nonimg: (871, 3)
Number of dynamic_fc samples: 871

Type of the first sample: <class 'torch_geometric.data.data.Data'>
Shape of node features x of the first sample: torch.Size([111, 111])
Shape of edge_index of the first sample: torch.Size([2, 5358])

Label distribution (label: count):
  0.0: 403
  1.0: 468

SITE_ID for all samples:
[ 0.  0.  0. ... 19. 19. 19.]

SEX for all samples:
[1. 2. 1. ... 1. 2. 1.]

AGE_AT_SCAN for all samples:
[37.7  20.2  20.9  ... 11.08  9.5  14.42]


## 2. Dataset splitting strategy in the current code

In this repository, dataset splitting is implemented in `dataload.py` via `dataloader.data_split()`. The **current implementation** adopts a site-wise leave-site-out cross-validation strategy, with the following key ideas:

- First, the original SITE_ID for each subject is read from the phenotypic file.
- Several sub-sites are **merged** into composite sites, for example:
  - UM_1 and UM_2 are merged into UM.
  - UCLA_1 and UCLA_2 are merged into UCLA.
  - Other sites keep their original names.
- On the merged site labels, the code selects the three largest target sites: NYU, UM, and UCLA.
- For each target site s ∈ {NYU, UM, UCLA}:
  - All samples from site s are used as the **test set**.
  - All samples from the remaining sites are used as the **training set**.

The following code cell calls `dataloader.data_split` from this directory and prints, for each fold, the number of samples and the site distribution in training and test sets, providing a clear view of the actual site-wise leave-group-out strategy used here.


In [3]:
# 2*. Detailed illustration of the site-wise leave-group-out strategy

from dataload import get_ids, get_subject_score

# Use the same opt and dl as above
cv_splits = dl.data_split(opt.n_folds)
n_folds = len(cv_splits)

labels = np.array(y)
sites = np.array(phonetic_score['SITE_ID'])  # integer-encoded site labels

# Recover mapping "site index -> original SITE_ID name" according to dataload.py
subject_list = get_ids()
site_dict = get_subject_score(subject_list, score='SITE_ID')  # {subject_id: SITE_NAME}
unique_site_names = np.unique(list(site_dict.values())).tolist()
site_id_to_name = {idx: name for idx, name in enumerate(unique_site_names)}

print(f'Total number of samples: {len(labels)}')
print(f'Number of effective folds n_folds: {n_folds}\n')

for fold, (train_idx, test_idx) in enumerate(cv_splits):
    y_tr, s_tr = labels[train_idx], sites[train_idx]
    y_te, s_te = labels[test_idx], sites[test_idx]

    print(f'====== Fold {fold} ======')
    print(f'Number of training samples: {len(train_idx)}, number of test samples: {len(test_idx)}')

    # Site distribution in training and test sets (using integer site indices)
    u_tr_site, c_tr_site = np.unique(s_tr, return_counts=True)
    u_te_site, c_te_site = np.unique(s_te, return_counts=True)

    print('Training set site distribution (SITE_ID index -> count):')
    print(dict(zip(u_tr_site.astype(int), c_tr_site)))
    print('Test set site distribution (SITE_ID index -> count):')
    print(dict(zip(u_te_site.astype(int), c_te_site)))

    # Additionally, print test-site name distribution (SITE_NAME -> count)
    test_site_name_counts = {}
    for sid, cnt in zip(u_te_site.astype(int), c_te_site):
        site_name = site_id_to_name.get(sid, f'UNKNOWN_{sid}')
        test_site_name_counts[site_name] = int(cnt)
    print('Test set site name distribution (SITE_NAME -> count):')
    print(test_site_name_counts)
    print()

    # If you also want to inspect joint (label, SITE_ID) distributions, you can uncomment the lines below:
    # comb_tr = y_tr * 1000 + s_tr
    # comb_te = y_te * 1000 + s_te
    # u_tr_comb, c_tr_comb = np.unique(comb_tr, return_counts=True)
    # u_te_comb, c_te_comb = np.unique(comb_te, return_counts=True)
    # print('Training set (label*1000 + SITE_ID) combination distribution:')
    # print(dict(zip(u_tr_comb.astype(int), c_tr_comb)))
    # print('Test set (label*1000 + SITE_ID) combination distribution:')
    # print(dict(zip(u_te_comb.astype(int), c_te_comb)))
    # print()


Total number of samples: 871
Number of effective folds n_folds: 3

====== Fold 0 ======
Number of training samples: 699, number of test samples: 172
Training set site distribution (SITE_ID index -> count):
{0: 15, 1: 11, 2: 33, 3: 28, 4: 28, 5: 46, 7: 25, 8: 28, 9: 50, 10: 26, 11: 27, 12: 25, 13: 44, 14: 64, 15: 21, 16: 86, 17: 34, 18: 67, 19: 41}
Test set site distribution (SITE_ID index -> count):
{6: 172}
Test set site name distribution (SITE_NAME -> count):
{'NYU': 172}

====== Fold 1 ======
Number of training samples: 751, number of test samples: 120
Training set site distribution (SITE_ID index -> count):
{0: 15, 1: 11, 2: 33, 3: 28, 4: 28, 5: 46, 6: 172, 7: 25, 8: 28, 9: 50, 10: 26, 11: 27, 12: 25, 13: 44, 14: 64, 15: 21, 18: 67, 19: 41}
Test set site distribution (SITE_ID index -> count):
{16: 86, 17: 34}
Test set site name distribution (SITE_NAME -> count):
{'UM_1': 86, 'UM_2': 34}

====== Fold 2 ======
Number of training samples: 786, number of test samples: 85
Training set s

## 3. Training example (calling main.py)

The official training and evaluation pipeline is implemented in `main.py` under `if __name__ == "__main__":`, which includes:

- Initializing hyperparameters (`OptInit`)
- Calling `dataloader` to load the data
- Running site-wise leave-group-out cross-validation
- Printing training and test metrics at each epoch
- Saving the best model (on the test set) for each fold to `opt.ckpt_path`

In this notebook we invoke `main.py` via a command-line call so that the behavior is exactly the same as running the script directly, and a **training log file** is automatically generated (handled by the `Logger` class).

> Training from scratch can be time-consuming, so in practice we usually pre-train on the server, save the best models and logs, and then use this notebook mainly for demonstration.


In [ ]:
import sys

# Use the Python interpreter of the current notebook kernel
!{sys.executable} main.py --train 1 --num_iter 500 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./run.log


In [4]:
# 3.2 View a snippet of the training log

log_path = './run.log'

if os.path.exists(log_path):
    print(f'Reading log file: {log_path}\n')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show only the last 40 lines for readability
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Log file does not exist. Please run the training cell above first.')


Reading log file: ./run.log

Epoch: 472,	ce loss: 0.06277,	ce loss_cla: 0.06277,	train acc: 0.98219,	test acc: 0.60000,	test spe: 0.45833,	test sen: 0.78378      2026-03-26 13:31:31
Epoch: 473,	ce loss: 0.04637,	ce loss_cla: 0.04637,	train acc: 0.98473,	test acc: 0.58824,	test spe: 0.45833,	test sen: 0.75676      2026-03-26 13:32:48
Epoch: 474,	ce loss: 0.04685,	ce loss_cla: 0.04685,	train acc: 0.98473,	test acc: 0.58824,	test spe: 0.45833,	test sen: 0.75676      2026-03-26 13:33:58
Epoch: 475,	ce loss: 0.05197,	ce loss_cla: 0.05197,	train acc: 0.98601,	test acc: 0.58824,	test spe: 0.45833,	test sen: 0.75676      2026-03-26 13:35:06
Epoch: 476,	ce loss: 0.09636,	ce loss_cla: 0.09636,	train acc: 0.97328,	test acc: 0.60000,	test spe: 0.47917,	test sen: 0.75676      2026-03-26 13:36:13
Epoch: 477,	ce loss: 0.05191,	ce loss_cla: 0.05191,	train acc: 0.97964,	test acc: 0.57647,	test spe: 0.43750,	test sen: 0.75676      2026-03-26 13:37:19
Epoch: 478,	ce loss: 0.05608,	ce loss_cla: 0.05608,	t

## 4. Evaluation with the best saved models

In `main.py`, the `evaluate()` function is responsible for:

- Loading the best-performing model for each fold from `opt.ckpt_path`.
- Running forward passes on the corresponding test set.
- Computing and printing multiple metrics (accuracy, AUC, sensitivity, specificity, F1-score, etc.).

Here we call the same script with `--train` set to `0` to perform evaluation using the saved models.

> Before evaluation, make sure that the corresponding model checkpoints for all folds have been successfully saved in `ckpt_path` during training.


In [5]:
import sys

# 4.1 Example: evaluate using pre-trained models

# Assume the training stage used ./ckpt_demo/ as ckpt_path.
# Here we keep the same ckpt_path, set train to 0, and run evaluation only.

!{sys.executable} main.py --train 0 --num_iter 500 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./test.log


 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is eval.
 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:500
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/test_data/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/test_data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/test_data/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> P

In [7]:
# 4.2 View a snippet of the evaluation log

log_path_test = './test.log'

if os.path.exists(log_path_test):
    print(f'Reading evaluation log file: {log_path_test}\n')
    with open(log_path_test, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show the last 40 lines of evaluation results
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Evaluation log file does not exist. Please run the evaluation cell above first.')


Reading evaluation log file: ./test.log

    (convs4): ModuleList(
      (0): TransformerConv(2000, 20, heads=1)
      (1-4): 4 x TransformerConv(40, 20, heads=1)
    )
    (bns): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns2): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns3): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (out_fc): Linear(in_features=100, out_features=2, bias=True)
    (conv5): Linear(in_features=20, out_features=1, bias=False)
    (conv6): Linear(in_features=20, out_features=1, bias=False)
    (conv7): Linear(in_features=20, out_features=1, bias=False)
    (conv8): Linear(in_features=20, out_features=1, bias=False)
    (conv9): Linear(in_features=20, out_features=1, bias=False)
    (conv10): Linear(in_features=20, out_features=1, bias=False)
 